# Initialization and Dataset

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv

if Path("open_ragbench").exists():
    print("All set!")
else:
    from huggingface_hub import snapshot_download
    snapshot_download(
        repo_id="vectara/open_ragbench",
        repo_type="dataset",
        local_dir="./open_ragbench"
    )
    print("Dataset ready!")

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")

In [ ]:
# Automatically finds all .json files in your directory
corpus_folder = Path("open_ragbench/pdf/arxiv/corpus")
all_json_files = list(corpus_folder.glob("*.json"))
# all_json_files = all_json_files[:50]

print(f"Found {len(all_json_files)} papers ready for indexing!")

# Indexing

In [ ]:
from haystack import Document, Pipeline
from haystack.components.writers import DocumentWriter

In [ ]:
from docling.chunking import HybridChunker
from haystack_integrations.components.converters.docling import DoclingConverter, ExportType
from haystack_integrations.document_stores.qdrant import QdrantDocumentStore
from haystack_integrations.components.embedders.vllm import VLLMDocumentEmbedder

In [ ]:
EXPORT_TYPE = ExportType.DOC_CHUNKS
# EMBED_MODEL_ID = "google/embeddinggemma-300m"
EMBED_MODEL_ID = "mlx-community/Qwen3-Embedding-0.6B-8bit"
GENERATION_MODEL_ID = "Qwen/Qwen3-0.6B"

In [ ]:
sources = []
for file in all_json_files:
    with open(file) as f:
        paper = json.load(f)

    for section in paper["sections"]:
        sources.append(
            Document(
                content=section["text"][:10],
                meta={
                    "title": paper["title"],
                    "authors": paper["authors"],
                    "abstract": paper["abstract"],
                    "paper_id": paper["id"],
                    "section_id": section["section_id"],
                    "categories": paper["categories"],
                    "updated": paper["updated"],
                    "published": paper["published"]
                }
            )
        )

In [ ]:
sources[0]

In [ ]:
indexing_pipeline = Pipeline()

document_store = QdrantDocumentStore(":memory:", embedding_dim=1024)

In [ ]:
converter = DoclingConverter(
    export_type=EXPORT_TYPE, 
    # chunker=HybridChunker(tokenizer=EMBED_MODEL_ID)
    )

indexing_embedder = VLLMDocumentEmbedder(model=EMBED_MODEL_ID,
                                         api_base_url="http://localhost:8001/v1")

writer = DocumentWriter(document_store=document_store)

In [ ]:
# indexing_pipeline.add_component("converter", converter)
indexing_pipeline.add_component("embedder", indexing_embedder)
# indexing_pipeline.connect("converter", "embedder")
indexing_pipeline.add_component("writer", writer)
indexing_pipeline.connect("embedder", "writer")

In [ ]:
indexing_pipeline.run(
    {
        # "converter": {"sources": sources},
        "embedder": {"documents": sources},
    })

In [ ]:
docs = document_store.filter_documents()

print(len(docs))

print(docs[0].embedding is not None)

print(type(docs[0].embedding))

print(len(docs[0].embedding) if docs[0].embedding else None)

print(docs[0].embedding[:10] if docs[0].embedding else None)

In [ ]:
from haystack.components.builders.chat_prompt_builder import ChatPromptBuilder
from haystack.dataclasses import ChatMessage
from haystack.components.builders import AnswerBuilder

In [ ]:
from haystack_integrations.components.retrievers.qdrant import QdrantEmbeddingRetriever

In [ ]:
from haystack_integrations.components.embedders.vllm import VLLMTextEmbedder
from haystack_integrations.components.generators.vllm import VLLMChatGenerator

In [ ]:
with open("open_ragbench/pdf/arxiv/queries.json") as f:
    queries = json.load(f)

all_questions = [queries[query]["query"] for query in queries]
# all_questions

with open("open_ragbench/pdf/arxiv/answers.json") as f:
    answers = json.load(f)

all_ground_truth_answers = [answers[answer] for answer in answers]
# all_ground_truth_answers

In [ ]:
chat_template = [
    ChatMessage.from_system(
        """
        You are a helpful AI assistant.
        Use the provided documents to answer the question.
        If the answer is not in the documents, say so.
        """
    ),
    ChatMessage.from_user(
        """
        Documents:
        {% for doc in documents %}
            {{ doc.content }}
        {% endfor %}

        Question:
        {{query}}
        """
    )
]

rag_pipeline = Pipeline()

In [ ]:
rag_embedder = VLLMTextEmbedder(model=EMBED_MODEL_ID,
                                api_base_url="http://localhost:8001/v1")

retriever = QdrantEmbeddingRetriever(document_store=document_store, top_k=5)

prompt_builder = ChatPromptBuilder(template=chat_template, required_variables={"query", "documents"})

llm = VLLMChatGenerator(model=GENERATION_MODEL_ID,
                        api_base_url="http://localhost:8000/v1")

answer_builder = AnswerBuilder()

In [ ]:
rag_pipeline.add_component("embedder", rag_embedder)
rag_pipeline.add_component("retriever", retriever)
rag_pipeline.connect("embedder.embedding", "retriever")
rag_pipeline.add_component("prompt_builder", prompt_builder)
rag_pipeline.connect("retriever", "prompt_builder")
rag_pipeline.add_component("llm", llm)
rag_pipeline.connect("prompt_builder", "llm")
rag_pipeline.add_component("answer_builder", answer_builder)
rag_pipeline.connect("llm", "answer_builder")
rag_pipeline.connect("retriever", "answer_builder")

In [ ]:
rag_pipeline.show()

In [ ]:
response = rag_pipeline.run(
    {
        "embedder": {"text": querys[0]},
        "prompt_builder": {"query": querys[0]},
        "answer_builder": {"query": querys[0]},
    })

In [ ]:
response["answer_builder"]["answers"][0].data